> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/02c_Local_Morans_I_Hotspots.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Where Are the Clusters? Local Moran's I (LISA)

**Series:** Spatial Autocorrelation with PySAL (notebook 3 of 3)
**Time:** about 30 minutes

**In this notebook you will:**
- Compute a Moran's I value for every county
- Keep only statistically significant results
- Make a hotspot map (High-High, Low-Low, outliers)
- Optional: repeat at the census tract level for Marion County

**You should already know:** notebooks 02a and 02b.

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## From global to local

Global Moran's I gave one number for all of Indiana. **Local Moran's I**, also called **LISA**
(Local Indicators of Spatial Association), gives one value per county and labels each one:

| Label | Meaning |
|---|---|
| **High-High (HH)** | High value surrounded by high values: a **hotspot** |
| **Low-Low (LL)** | Low value surrounded by low values: a **coldspot** |
| **High-Low (HL)** | High value surrounded by low values: an outlier |
| **Low-High (LH)** | Low value surrounded by high values: an outlier |
| **Not significant** | No clear local pattern |

## Step 1: Import

In [ ]:
# libpysal esda splot mapclassify comes installed in the SpatialTurn GIS kernel, so there is nothing to install.

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

## Step 2: Get the data

We use the **American Community Survey (ACS) 5-year estimates, 2023** for Indiana's 92 counties:

| Census variable | Meaning | Our column |
|---|---|---|
| `B01003_001E` | Total population | `total_pop` |
| `B01002_001E` | Median age | `median_age` |
| `B02001_005E` | Asian alone population | `asian_pop` |

> **Every learner needs their own free Census API key.** It is the same key used in the Census data
> session. If you do not have one yet, request it at <https://api.census.gov/data/key_signup.html>;
> it arrives by email within a few minutes and must be activated from the link in that email.

Paste your key between the quotes below. The data and the county boundaries are read straight from
the Census website into memory, so there is nothing to download.

In [ ]:
CENSUS_KEY = "PASTE_YOUR_CENSUS_API_KEY_HERE"

def load_indiana_counties(key):
    if key.startswith("PASTE"):
        raise ValueError("Paste your free Census API key into CENSUS_KEY above, then run this cell again.")
    # 1. Attributes from the Census API
    reply = requests.get(
        "https://api.census.gov/data/2023/acs/acs5",
        params={
            "get": "NAME,B01003_001E,B01002_001E,B02001_005E",
            "for": "county:*",
            "in": "state:18",          # 18 = Indiana
            "key": key,
        },
        timeout=60,
    )
    reply.raise_for_status()
    rows = reply.json()
    table = pd.DataFrame(rows[1:], columns=rows[0])
    table["GEOID"] = table["state"] + table["county"]
    table = table.rename(columns={"B01003_001E": "total_pop",
                                  "B01002_001E": "median_age",
                                  "B02001_005E": "asian_pop"})
    for col in ["total_pop", "median_age", "asian_pop"]:
        table[col] = pd.to_numeric(table[col])
    table["pct_asian"] = 100 * table["asian_pop"] / table["total_pop"]

    # 2. County boundaries from the Census cartographic boundary files
    shapes = gpd.read_file(
        "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip")
    shapes = shapes[shapes["STATEFP"] == "18"][["GEOID", "geometry"]]

    # 3. Join, then project to meters (UTM zone 16N) for spatial analysis
    counties = shapes.merge(table, on="GEOID").to_crs(epsg=26916)
    return counties.reset_index(drop=True)

counties = load_indiana_counties(CENSUS_KEY)
print(len(counties), "counties loaded")
counties[["NAME", "total_pop", "median_age", "pct_asian"]].head()

## Step 3: Build weights and run Local Moran's I

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran_Local

w = Queen.from_dataframe(counties, use_index=True)
w.transform = "R"

lisa = Moran_Local(counties["median_age"], w, permutations=999, seed=42)

## Step 4: Label each county

`lisa.q` gives the quadrant (1 = HH, 2 = LH, 3 = LL, 4 = HL) and `lisa.p_sim` the p-value.
We only trust labels where p is below 0.05.

In [ ]:
labels = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
counties["lisa"] = [labels[q] for q in lisa.q]
counties.loc[lisa.p_sim >= 0.05, "lisa"] = "Not significant"

counties["lisa"].value_counts()

Most counties are usually "Not significant". That is normal and important: without the
significance filter you would report many false hotspots.

## Step 5: Map the clusters

In [ ]:
from splot.esda import lisa_cluster

fig, ax = lisa_cluster(lisa, counties, p=0.05, figsize=(6, 8))
ax.set_title("Median age: Local Moran's I clusters (p < 0.05)")
plt.show()

Red counties are older areas next to older areas. Blue are younger areas next to younger areas
(often around cities and universities). Light colors are outliers.

## Step 6: Which counties are hotspots?

In [ ]:
counties.loc[counties["lisa"] != "Not significant", ["NAME", "median_age", "lisa"]] \
    .sort_values("lisa")

## Step 7 (optional, advanced): Census tracts in Marion County

Counties are large. Running the same analysis on smaller units, such as census tracts in Marion County
(Indianapolis), often shows patterns that county maps hide.

In [ ]:
def load_marion_tracts(key):
    if key.startswith("PASTE"):
        raise ValueError("Paste your free Census API key into CENSUS_KEY, then run this cell again.")
    reply = requests.get(
        "https://api.census.gov/data/2023/acs/acs5",
        params={"get": "NAME,B01003_001E,B01002_001E",
                "for": "tract:*", "in": ["state:18", "county:097"], "key": key},
        timeout=60,
    )
    reply.raise_for_status()
    rows = reply.json()
    table = pd.DataFrame(rows[1:], columns=rows[0])
    table["GEOID"] = table["state"] + table["county"] + table["tract"]
    table = table.rename(columns={"B01003_001E": "total_pop", "B01002_001E": "median_age"})
    table[["total_pop", "median_age"]] = table[["total_pop", "median_age"]].apply(pd.to_numeric)
    table = table[(table["total_pop"] > 0) & (table["median_age"] > 0)]  # drop empty tracts

    shapes = gpd.read_file(
        "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_18_tract_500k.zip")
    shapes = shapes[shapes["COUNTYFP"] == "097"][["GEOID", "geometry"]]
    return shapes.merge(table, on="GEOID").to_crs(epsg=26916).reset_index(drop=True)

tracts = load_marion_tracts(CENSUS_KEY)
print(len(tracts), "tracts")

In [ ]:
w_t = Queen.from_dataframe(tracts, use_index=True)
w_t.transform = "R"
lisa_t = Moran_Local(tracts["median_age"], w_t, permutations=999, seed=42)

fig, ax = lisa_cluster(lisa_t, tracts, p=0.05, figsize=(8, 8))
ax.set_title("Marion County tracts: median age clusters")
plt.show()

## Try it yourself

Make a LISA cluster map of **percent Asian** (`pct_asian`) for Indiana counties. Where are the High-High clusters?

<details>
<summary><b>Show a solution</b></summary>

```python
lisa_asian = Moran_Local(counties["pct_asian"], w, permutations=999, seed=42)
fig, ax = lisa_cluster(lisa_asian, counties, p=0.05, figsize=(6, 8))
ax.set_title("Percent Asian: LISA clusters")
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- Local Moran's I (LISA) gives a value and a cluster label for every place.
- Always filter by p-value; unfiltered LISA maps show false hotspots.
- High-High are hotspots, Low-Low are coldspots, HL and LH are outliers.
- Results change with the size of the units (counties vs tracts).